# Modelos de regresión en SKlearn
## 1. Objetivo

Familiarizarse con los conceptos validación cruzada

## Datos de Melb Houses
* Suburb: Suburb
* Address: Address
* Rooms: Number of rooms
* Price: Price in Australian dollars
* Method:
  * S - property sold;
  * SP - property sold prior;
  * PI - property passed in;
  * PN - sold prior not disclosed;
  * SN - sold not disclosed;
  * NB - no bid;
  * VB - vendor bid;
  * W - withdrawn prior to auction;
  * SA - sold after auction;
  * SS - sold after auction price not disclosed.
  * N/A - price or highest bid not available.

* Type:
  * br - bedroom(s);
  * h - house,cottage,villa, semi,terrace;
  * u - unit, duplex;
  * t - townhouse;
  * dev site - development site;
  * res - other residential.

* SellerG: Real Estate Agent

* Date: Date sold

* Distance: Distance from CBD in Kilometres
* Regionname: General Region (West, North West, North, North east …etc)
* Propertycount: Number of properties that exist in the suburb.
* Bedroom2 : Scraped # of Bedrooms (from different source)
* Bathroom: Number of Bathrooms
* Car: Number of carspots
* Landsize: Land Size in Metres
* BuildingArea: Building Size in Metres
* YearBuilt: Year the house was built
* CouncilArea: Governing council for the area
* Lattitude: Self explanitory
* Longtitude: Self explanitory


## 2. Librerias de trabajo

In [ ]:
# Instala libreria Pandas si no la tenemos
#pip install pandas seaborn scikit-learn -y

In [2]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import (
    LinearRegression,
    Lasso,
    Ridge,
    ElasticNet
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import (
    LabelEncoder,
    OrdinalEncoder,
    LabelBinarizer,
    OneHotEncoder
)

from sklearn.linear_model import (
    LinearRegression,
    Lasso,
    Ridge,
    ElasticNet
)

from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

## 3. Lectura de datos

Primero nos encargaremos de leer los datos, indicando a Python donde se encuentra la carpeta que contiene los datos y los nombres de los archivos relevantes para el análisis.

In [3]:
#  Indicamos la ruta a la carpeta de de tu computadora 
# donde se ubican los datos del E-commerce
# Ejemplo: "C:\Usuarios\[tu nombre]\Descargas"

DATA_PATH="/Users/cesar/sandbox/ai_programming_foundations/data"

Ahora procederemos a definir una variable que indique el nombre del archivo junto con su extensión (por ejemplo, `.csv`):

In [4]:
FILE_DATA_PATH = "melb_data.csv"

Echaremos mano de la utilidad `os.path.join` de Python que indicar rutas en tu computadora donde se ubican archivos, así Pandas encontrá los archivos de datos.


**Ejemplo**

A continuación mostraremos un ejemplo leyendo el archivo `melb_data.csv`:

In [5]:
# Ejemplo
print(f"Ruta del archivo: {FILE_DATA_PATH}")
print(os.path.join(DATA_PATH, FILE_DATA_PATH))

Ruta del archivo: melb_data.csv
/Users/cesar/sandbox/ai_programming_foundations/data/melb_data.csv


In [25]:
# Leemos con pandas
df = pd.read_csv(
    os.path.join(DATA_PATH, FILE_DATA_PATH)
    )

In [26]:
df.head(10)

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000,S,Biggin,03/12/16,2.5,3067,...,1,1.0,202,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019
1,Abbotsford,25 Bloomburg St,2,h,1035000,S,Biggin,04/02/16,2.5,3067,...,1,0.0,156,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019
2,Abbotsford,5 Charles St,3,h,1465000,SP,Biggin,04/03/17,2.5,3067,...,2,0.0,134,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019
3,Abbotsford,40 Federation La,3,h,850000,PI,Biggin,04/03/17,2.5,3067,...,2,1.0,94,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019
4,Abbotsford,55a Park St,4,h,1600000,VB,Nelson,04/06/16,2.5,3067,...,1,2.0,120,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019
5,Abbotsford,129 Charles St,2,h,941000,S,Jellis,07/05/16,2.5,3067,...,1,0.0,181,NaN,NaN,Yarra,-37.8041,144.9953,Northern Metropolitan,4019
6,Abbotsford,124 Yarra St,3,h,1876000,S,Nelson,07/05/16,2.5,3067,...,2,0.0,245,210.0,1910.0,Yarra,-37.8024,144.9993,Northern Metropolitan,4019
7,Abbotsford,98 Charles St,2,h,1636000,S,Nelson,08/10/16,2.5,3067,...,1,2.0,256,107.0,1890.0,Yarra,-37.8060,144.9954,Northern Metropolitan,4019
8,Abbotsford,6/241 Nicholson St,1,u,300000,S,Biggin,08/10/16,2.5,3067,...,1,1.0,0,NaN,NaN,Yarra,-37.8008,144.9973,Northern Metropolitan,4019
9,Abbotsford,10 Valiant St,2,h,1097000,S,Biggin,08/10/16,2.5,3067,...,1,2.0,220,75.0,1900.0,Yarra,-37.8010,144.9989,Northern Metropolitan,4019


## 5. Validación Cruzada y selección del mejor modelo

Como hemos visto, al presentarse modelos con mayor complejidad, estos suelen acompañarse por **hiper-parámetros**, es decir, parámetros que no se ajustan automáticamente durante el entrenamiento y necesitan ser especificados de antemano. Ello representa un dilema, pues no tenemos forma de saber como se desempeñara un modelo ante datos que nunca ha visto.

Para ello, existe una técnica denominada **validación cruzada**, que, en términos sencillos, consiste en dividir los datos en varios conjuntos de entrenamiento y prueba, y entrenar y evaluar el modelo varias veces utilizando diferentes combinaciones de estos conjuntos. Esto nos da una mejor estimación de la capacidad del modelo para generalizar a datos nuevos y no vistos.

Esta técnica se utiliza en la calibración de hiperparámetros para evitar el sobreajuste y seleccionar los mejores valores de los hiperparámetros que maximicen el rendimiento en los datos no vistos durante la evaluación. Es decir, al tener varios conjuntos de entrenamiento y prueba, se puede aproximar el comportamiento de una modelo a través de diferentes combinaciones de valores de sus hiper-parámetros y con ello selección los que tienen un mayor desempeño, que típicamente se obtiene promediando el valor de la función de pérdida en todos los subconjuntos de entrenamiento y prueba generados y tomando el que tiene mejor desempeño.

En la práctica la construcción de los conjuntos de entrenamiento para validación cruzada esencialmente se basa en tomar construir subconjuntos disjuntos del conjunto de entrenamiento. Básicamente estos subconjuntos se toman como una nueva versión de datos para entrenar y probar modelos, cuidando que no sucede fuga de datos.

**Figura 1:** *Esquema de los datos de validación cruzada con k-hojas .*
![title](https://scikit-learn.org/stable/_images/sphx_glr_plot_cv_indices_006.png)


En Python, existe una clase que nos permite construir esta clase de conjuntos de validación llamada `KFold`. El resto de estrategias de validación cruzada se puede consultar en la documentación de SKlearn https://scikit-learn.org/stable/modules/cross_validation.html

In [15]:
import numpy as np
from sklearn.model_selection import KFold

X = ["a", "b", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l"]
kf = KFold(n_splits=5)
for train, test in kf.split(X):
    print("%s %s" % (train, test))

[ 3  4  5  6  7  8  9 10 11] [0 1 2]
[ 0  1  2  6  7  8  9 10 11] [3 4 5]
[ 0  1  2  3  4  5  8  9 10 11] [6 7]
[ 0  1  2  3  4  5  6  7 10 11] [8 9]
[0 1 2 3 4 5 6 7 8 9] [10 11]


**Nota:**

* Estos son los indices de como vamos a particionar los datos para ejecutar la validacion cruzada

## 5.1 Calibración de hiper parámetros con validación cruzada

Permite realizar la calibración de hiper parámetros de una manera sencilla empleando la clase `GridSearchCV` (véase https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), ya que unicamente tenemos que pasarle:
* el modelo o el pipeline que queremos ajustar,
* un diccionario indicando los hiper parámetros del modelo (con un formato específico, a discutir en breve)
* el esquema de validación cruzada (en nuestro caso es la instancia de `KFold`)
* el nombre métrica necesaria para evaluar el ajuste (ver seccion **Regression** de https://scikit-learn.org/stable/modules/model_evaluation.html).

Y con ello se encargará de probar todas las combinaciones de parámetros que le proporcionemos para regresarnos posteriormente el modelo con el mejor desempeño en los subconjuntos de validación cruzada. Dicho modelo, se debe evaluar aun en el conjunto de prueba ya con los mejores hiper-parámetros elegidos en el proceso de validación cruzada.




### 5.1.1 **Ejemplo de validación cruzada**

En este ejemplo ajustaremos un modelo de regresión de Lasso, como sabemos toma un hiper parámetros de nombre alpha, con la métrica RMSE.



In [17]:
# importamos la libreria GridSearchCV
from sklearn.model_selection import GridSearchCV

Definimos el modelo del que queremos ajustar los hiper parámetros en el conjunto de entrenamiento usando validación cruzada, en este caso es una regresión de Lasso:

In [39]:
# modelo de regresion de laso
reg_lasso = Lasso(
    alpha=0.0001 ,
    max_iter= 3000,
    random_state=0
    )

Ahora definimos un diccionario con los valores del hiper parámetro a probar:

In [29]:
parameters = {
    'alpha': [
        1e-15,
        1e-13,
        1e-10,
        1e-8,
        1e-5,
        1e-4,
        1e-3,
        1e-2,
        1e-1,
        1,
        5,
        10,
        20,
        30,
        40,
        45,
        50,
        55,
        60,
        100,
        0.0014]
    }

Escogemos el nombre de la métrica, en este caso es RSME, denominada como `neg_root_mean_squared_error` en la documentación https://scikit-learn.org/stable/modules/model_evaluation.html.

Por otro lado, creamos el conjunto de indices para la validación cruzada, indicando el número de conjunto de entrenamiento, así como el tamaño del conjunto de entrenamiento (1440):

In [30]:
n_splits = 5
cv = KFold(n_splits=n_splits)

Ahora entrenamos el modelo y lo evaluamos:

In [32]:
# Define listas de columnas que van a emplearse en el modelado
num_features = [
    'Rooms', 
    'BuildingArea',
    'Landsize',
    'Distance',
    'Bathroom',
    'YearBuilt'
 ]

cat_cols = ['Regionname', 'Type']

# Lista que tiene todas los grupos de columnas
non_target_cols = num_features + cat_cols

target = ['Price']

In [33]:
df["BuildingArea"] = df["BuildingArea"].fillna(0)

In [34]:
df["YearBuilt"] = df["YearBuilt"].fillna(df["YearBuilt"].median())

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    df[non_target_cols],
    df[target],
    test_size=0.2,
)

Comunicamos esta información a GridSearchCV

In [43]:
model_lasso = GridSearchCV(
    reg_lasso,
    parameters,
    n_jobs=-1,
    scoring='neg_mean_squared_error',
    cv=cv)

In [45]:
model_lasso.fit(X_train[num_features], y_train)

,estimator,Lasso(alpha=0...andom_state=0)
,param_grid,"{'alpha': [1e-15, 1e-13, ...]}"
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,KFold(n_split...shuffle=False)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,1e-10


Evaluando el modelo:

In [48]:
y_train_pred = model_lasso.predict(X_train[num_features])
y_test_pred = model_lasso.predict(X_test[num_features])

# error en conjunto de entrenamiento y prueba
error_train = root_mean_squared_error(y_train, y_train_pred)
error_test = root_mean_squared_error(y_test, y_test_pred)

# errores
print("Error RSME en train:", round(error_train,2) )
print("Error RSME en test:", round(error_test,2) )

Error RSME en train: 477761.48
Error RSME en test: 494136.39


También podemos acceder a lo resultados de la validación cruzada:

In [51]:
print(" Resultados" )
print("\n Mejor modelo en calibración :\n", model_lasso.best_estimator_)
print("\n Mejor métrica de evaluación:\n", model_lasso.best_score_)
print("\n Parámetro con mejor desempeño:\n", model_lasso.best_params_)


 Resultados

 Mejor modelo en calibración :
 Lasso(alpha=1e-10, max_iter=3000, random_state=0)

 Mejor métrica de evaluación:
 -238339275123.18158

 Parámetro con mejor desempeño:
 {'alpha': 1e-10}


**Preguntas**
* ¿Cuál fue el hiper parámetro que dió el mejor modelo?

### 5.1.2 Ejemplo de validación cruzada usando Pipeline

La validación cruzada también de puede aplicar usando un procesamiento de datos con Pipeline, el flujo casi igual al anterior, con la excepción de que la espeficicación del diccinario que indar el hiper parámetro se modificará un poco: se deberá añadir el nombre del modelo con un sufijo de doble guión bajo.

Ahora veremos un ejemplo de este flujo, empleando el ejemplo del notebook anterior:

**Importamos librerias**

In [52]:
from sklearn.feature_selection import SelectKBest, r_regression
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

**Definimos las columnas del modelo**

In [53]:
# Define listas de columnas que van a emplearse en el modelado
num_features = [
    'Rooms', 
    'BuildingArea',
    'Landsize',
    'Distance',
    'Bathroom',
    'YearBuilt'
 ]

cat_cols = ['Regionname', 'Type']

# Lista que tiene todas los grupos de columnas
non_target_cols = num_features + cat_cols

target = ['Price']

**Creamos el pipeline de pre-procesamiento**

Notas:
    * En `pipe_standard_ohe` se ha definido un modelo Lasso con nombre `model`
    * En `model__alpha` se ha espeficido el valor del parámetro usando `model__alpha`, es decir agregamos al inicio el nombre que le dimos al modelo en el pipeline más doble guion bajo.

In [56]:
# Pipeline para escalar con estandar z-score
numerical_pipe = Pipeline([
    ('standar_scaler', StandardScaler())
])

# Pipeline para aplicar one hot encoding
categorical_pipe = Pipeline([
    ('one_hot', OneHotEncoder(handle_unknown='ignore'))
])

# Combina ambos procesos en columnas espeficadas en listas
pre_processor = ColumnTransformer([
    ('numerical', numerical_pipe, num_features),
    ('categorical', categorical_pipe, cat_cols),
], remainder='passthrough')

# comunica al pipeline la lista en el orden que se deben aplicar
# estos pasos

pipe_standard_ohe = Pipeline([
    ('transform', pre_processor),
    # Define modelo lasso
    ('model', Lasso(alpha=0.0001 , max_iter= 3000, random_state=0))
])

# Diccionario con el nombre del modelo
param_grid = {
    'model__alpha': [1e-15,1e-13,1e-10,1e-8,1e-5,1e-4,1e-3,1e-2,1e-1,1,5,10,20,30,40,45,50,55,60,100,0.0014]
}

lasso_reg_2 = GridSearchCV(
    pipe_standard_ohe,
    param_grid,
    n_jobs=-1,
    scoring='neg_mean_squared_error',
    cv=cv
    )

In [57]:
# Realiza la transformacion de los datos y el ajuste del modelo
lasso_reg_2.fit(X_train[non_target_cols], y_train)

,estimator,Pipeline(step...om_state=0))])
,param_grid,"{'model__alpha': [1e-15, 1e-13, ...]}"
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,KFold(n_split...shuffle=False)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('numerical', ...), ('categorical', ...)]"


**Evaluamos el modelo**

In [59]:
y_train_pred = lasso_reg_2.predict(X_train[non_target_cols])
y_test_pred = lasso_reg_2.predict(X_test[non_target_cols])

# error en conjunto de entrenamiento y prueba
error_train = root_mean_squared_error(y_train, y_train_pred)
error_test = root_mean_squared_error(y_test, y_test_pred)

# errores
print("Error RSME en train:", round(error_train,2) )
print("Error RSME en test:", round(error_test,2) )

Error RSME en train: 405357.11
Error RSME en test: 428741.18


In [60]:
print(" Resultados" )
print("\n Mejor modelo en calibración :\n", lasso_reg_2.best_estimator_)
print("\n Mejor métrica de evaluación:\n", lasso_reg_2.best_score_)
print("\n Parámetro con mejor desempeño:\n", lasso_reg_2.best_params_)

 Resultados

 Mejor modelo en calibración :
 Pipeline(steps=[('transform',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numerical',
                                                  Pipeline(steps=[('standar_scaler',
                                                                   StandardScaler())]),
                                                  ['Rooms', 'BuildingArea',
                                                   'Landsize', 'Distance',
                                                   'Bathroom', 'YearBuilt']),
                                                 ('categorical',
                                                  Pipeline(steps=[('one_hot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Regionname', 'Type'])])),
                ('model', Lasso(alpha=10, max_iter=3000, random_state

### 5.1.1 **Ejemplo de validación cruzada con Pipeline y más de un modelo**

Ahora mostraremos con la estructura anterior se puede usar para ajustar los hiper-parámetros de varios modelos a la vez. El código es casi igual, solo hay que especificar los modelos y sus hiper parámetros en forma particular.

**Especificamos el pipeline de procesamiento**

Aquí indicaremos un modelo cualquiera

In [62]:
from sklearn.ensemble import RandomForestRegressor

In [63]:
# Pipeline para escalar con estandar z-score
numerical_pipe = Pipeline([
    ('standar_scaler', StandardScaler()),
])

# Pipeline para aplicar one hot encoding
categorical_pipe = Pipeline([
    ('one_hot', OneHotEncoder(handle_unknown='ignore'))
])

# Combina ambos procesos en columnas espeficadas en listas
pre_processor = ColumnTransformer([
    ('numerical', numerical_pipe, num_features),
    ('categorical', categorical_pipe, cat_cols),
], remainder='passthrough')

# comunica al pipeline la lista en el orden que se deben aplicar
# estos pasos

pipe_standard_ohe = Pipeline([
    ('pre_processro', pre_processor),
    # Define modelo lasso
    ('model', RandomForestRegressor())
])

**Especificaciones de modelos y parámetros**

Definimos los modelos a probar:

In [64]:
model1 =  Lasso()
model2 = RandomForestRegressor()

*Modelo 1: Regresión de Lasso*

Especificamos los hiper-parámetros del modelo y la instancia del modelo lasso

In [65]:
params1 = {}
params1['model__alpha'] = [1e-15,1e-13,1e-10,1e-8,1e-5,1e-4,1e-3,1e-2,1e-1,1,5,10,20,30,40,45,50,55,60,100,0.0014]
params1['model__max_iter'] = [1500, 3000]
params1['model'] = [model1] # <- modelo dentro de una lista

**Modelo 2: Bosque Aleatorio**

Nuevamente especificamos los hiper-parámetros del modelo y la instancia del modelo. Los hiper parámetros se pueden consultar aqui: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html#sklearn.ensemble.RandomForestRegressor

In [66]:
params2 = {}
params2['model__n_estimators'] = [3, 4, 5, 10],
params2['model__max_features'] = ["auto", "sqrt", "log2"]
params2['model__min_samples_split'] = [3, 4, 5, 10]
params2['model__bootstrap'] = [True, False]
params2['model'] = [model2] # <- modelo dentro de una lista

**Ahora generamos una lista de los diccionarions de parámetros y modelos**

In [67]:
params_multi = [params1, params2]

In [68]:
model_csv_multi = GridSearchCV(
    pipe_standard_ohe,
    params_multi,
    n_jobs=-1,
    scoring='neg_mean_squared_error',
    cv=cv
    )

In [69]:
# Realiza la transformacion de los datos y el ajuste del modelo
model_csv_multi.fit(X_train[non_target_cols], y_train)


,estimator,Pipeline(step...Regressor())])
,param_grid,"[{'model': [Lasso()], 'model__alpha': [1e-15, 1e-13, ...], 'model__max_iter': [1500, 3000]}, {'model': [RandomForestRegressor()], 'model__bootstrap': [True, False], 'model__max_features': ['auto', 'sqrt', ...], 'model__min_samples_split': [3, 4, ...], ...}]"
,scoring,'neg_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,KFold(n_split...shuffle=False)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('numerical', ...), ('categorical', ...)]"


**Evaluamos el modelo**

In [70]:
y_train_pred = model_csv_multi.predict(X_train[non_target_cols])
y_test_pred = model_csv_multi.predict(X_test[non_target_cols])

# error en conjunto de entrenamiento y prueba
error_train = root_mean_squared_error(y_train, y_train_pred)
error_test = root_mean_squared_error(y_test, y_test_pred)

# errores
print("Error RSME en train:", round(error_train,2) )
print("Error RSME en test:", round(error_test,2) )

Error RSME en train: 405357.58
Error RSME en test: 428743.87


In [71]:
print(" Resultados" )
print("\n Mejor modelo en calibración :\n", model_csv_multi.best_estimator_)
print("\n Mejor métrica de evaluación:\n", model_csv_multi.best_score_)
print("\n Parámetro con mejor desempeño:\n", model_csv_multi.best_params_)

 Resultados

 Mejor modelo en calibración :
 Pipeline(steps=[('pre_processro',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numerical',
                                                  Pipeline(steps=[('standar_scaler',
                                                                   StandardScaler())]),
                                                  ['Rooms', 'BuildingArea',
                                                   'Landsize', 'Distance',
                                                   'Bathroom', 'YearBuilt']),
                                                 ('categorical',
                                                  Pipeline(steps=[('one_hot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Regionname', 'Type'])])),
                ('model', Lasso(alpha=20, max_iter=1500))])

 Mej

**Pregunta**
* ¿cual modelo tuvo desempeño?
* Modifica estos codigos para tu proyecto!